<a href="https://colab.research.google.com/github/lis-r-barreto/llm-study-notes/blob/main/hugging-face-llm-course/Chapter_1_Section_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformers, what can they do?

This notebook demonstrates various Natural Language Processing (NLP) tasks using the Hugging Face Transformers library. It covers common applications such as sentiment analysis, zero-shot classification, text generation, mask filling, named entity recognition, and question answering. The goal is to provide a hands-on introduction to using pre-trained models and pipelines for different NLP challenges.

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

Before running the code, we need to install the necessary libraries: `transformers`, `datasets`, and `evaluate`. We also specify `transformers[sentencepiece]` to ensure all dependencies for various tokenizers are included.

In [ ]:
!pip --quiet install datasets evaluate transformers[sentencepiece]

### Sentiment Analysis

Sentiment analysis is the task of classifying the emotional tone behind a piece of text. The `pipeline` function provides a high-level API to quickly perform this task using a pre-trained model. By default, if no model is specified, it uses a sentiment analysis model fine-tuned on the SST-2 dataset.

The most basic object in the 🤗 Transformers library is the pipeline() function. It connects a model with its necessary preprocessing and postprocessing steps, allowing us to directly input any text and get an intelligible answer:

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
classifier("I've been waiting for a HuggingFace course my whole life.")

The pipeline can also handle multiple inputs, processing them in a batch to return sentiment labels and scores for each sentence.

We can even pass several sentences!

In [ ]:
classifier(
    ["I've been waiting for a HuggingFace course my whole life.", "I hate this so much!"]
)

### Zero-shot Classification

Zero-shot classification allows you to classify texts into categories that the model has not been explicitly trained on. Instead of fixed labels, you provide a list of `candidate_labels`, and the model determines which label best fits the input text. This is particularly useful when you have custom categories or limited labeled data.

## Zero-shot classification

We’ll start by tackling a more challenging task where we need to classify texts that haven’t been labelled. This is a common scenario in real-world projects because annotating text is usually time-consuming and requires domain expertise. For this use case, the zero-shot-classification pipeline is very powerful: it allows you to specify which labels to use for the classification, so you don’t have to rely on the labels of the pretrained model. You’ve already seen how the model can classify a sentence as positive or negative using those two labels — but it can also classify the text using any other set of labels you like.

In [ ]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"],
)

Here, we're classifying a text about machine learning deployment into a custom set of labels related to 'mlops', 'analytics', or 'business'.

 ✏️ Try it out! Play around with your own sequences and labels and see how the model behaves.

In [ ]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification")
classifier(
    "Developing efficient workflows for deploying, monitoring, and maintaining machine learning models in production environments is a critical aspect of our strategy.",
    candidate_labels=["mlops", "analytics", "business"],
)

### Text Generation

Text generation models can create coherent and contextually relevant text based on a given prompt. The `text-generation` pipeline uses a language model (e.g., GPT-2 by default) to predict the next words in a sequence.

## Text generation

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation")
generator("In this course, we will teach you how to")

We can control the length of the generated text and the number of distinct sequences returned using `max_length` and `num_return_sequences` parameters.

✏️ Try it out! Use the num_return_sequences and max_length arguments to generate two sentences of 15 words each.

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation")
generator(
    "In this course, we will teach you how to",
    num_return_sequences=2,
    max_length=15,
)

You can also specify a particular model from the Hugging Face Hub for text generation, such as `distilgpt2`, which is a smaller, faster version of GPT-2.

## Using any model from the Hub in a pipeline

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="distilgpt2")
generator(
    "In this course, we will teach you how to",
    max_length=30,
    num_return_sequences=2,
)

This example demonstrates how to perform text generation for a different language (Portuguese) by directly loading a specific tokenizer and model (`pierreguillou/gpt2-small-portuguese`). This manual approach is sometimes necessary when the `pipeline` function might not directly support a specific model or task configuration, or when finer control over the tokenization and generation process is needed.

✏️ Try it out! Use the filters to find a text generation model for another language. Feel free to play with the widget and use it in a pipeline!

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained("pierreguillou/gpt2-small-portuguese")
model = AutoModelForCausalLM.from_pretrained("pierreguillou/gpt2-small-portuguese")

# Get sequence length max of 1024
tokenizer.model_max_length=1024

model.eval()  # disable dropout (or leave in train mode to finetune)

# input sequence
text = "Quem era Jim Henson? Jim Henson era um"
inputs = tokenizer(text, return_tensors="pt")

# Generate output
outputs = model.generate(inputs.input_ids, max_length=len(inputs.input_ids[0]) + 5, num_return_sequences=1, pad_token_id=tokenizer.eos_token_id)
predicted_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# For simplicity, just print the generated sequence, as the original code was trying to extract a single token prediction.
print('input text:', text)
print('predicted text:', predicted_text)


### Mask Filling

Mask filling (also known as masked language modeling) is a task where the model predicts masked tokens in a sequence. This is useful for understanding the context of words and can be used for tasks like grammar correction or data augmentation. The `fill-mask` pipeline takes a sentence with a `<mask>` token and suggests possible words for that mask.

## Mask filling

In [ ]:
from transformers import pipeline

unmasker = pipeline("fill-mask")
unmasker("This course will teach you all about <mask> models.", top_k=2)

The `top_k` parameter allows us to specify how many of the most likely predictions for the masked token should be returned.

✏️ Try it out! Search for the bert-base-cased model on the Hub and identify its mask word in the Inference API widget. What does this model predict for the sentence in our pipeline example above?

### Named Entity Recognition (NER)

Named Entity Recognition (NER) is the task of identifying and classifying named entities in text into pre-defined categories such as person names, organizations, locations, medical codes, time expressions, quantities, monetary values, percentages, etc. The `ner` pipeline can extract these entities.

## Named entity recognition

In [ ]:
from transformers import pipeline

ner = pipeline("ner", aggregation_strategy="simple")
ner("My name is Sylvain and I work at Hugging Face in Brooklyn.")

The `aggregation_strategy='simple'` parameter combines consecutive tokens that belong to the same entity (e.g., 'Hugging' and 'Face' become 'Hugging Face' with one 'ORG' label).

## Question answering

### Question Answering

Question Answering (QA) involves extracting an answer to a question from a given context. Unlike some of the other tasks, this section demonstrates a manual approach to question answering, directly using `AutoTokenizer` and `AutoModelForQuestionAnswering`. This method was chosen to bypass previous issues encountered with the `pipeline` abstraction for this specific task.

Here, the model identifies the most probable start and end tokens of the answer within the provided context based on the question.

In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-cased-distilled-squad")
model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-cased-distilled-squad")

question = "Where do I work?"
context = "My name is Sylvain and I work at Hugging Face in Brooklyn"

inputs = tokenizer(question, context, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

answer_start_index = outputs.start_logits.argmax()
answer_end_index = outputs.end_logits.argmax()

predict_answer_tokens = inputs.input_ids[0, answer_start_index : answer_end_index + 1]
answer = tokenizer.decode(predict_answer_tokens, skip_special_tokens=True)

print(f"Question: {question}")
print(f"Context: {context}")
print(f"Answer: {answer}")

## Resources

Here are some valuable resources to help you further explore the Hugging Face Transformers library and related topics.

### Official Documentation

*   **Hugging Face Transformers Documentation:** The official documentation is the best place to find detailed information on using the library, including API references, tutorials, and guides.
    *   [https://huggingface.co/docs/transformers/index](https://huggingface.co/docs/transformers/index)
*   **Hugging Face Models:** Explore a vast collection of pre-trained models available on the Hugging Face Hub.
    *   [https://huggingface.co/models](https://huggingface.co/models)
*   **Hugging Face Datasets:** Discover and use various datasets for training and evaluating your models.
    *   [https://huggingface.co/datasets](https://huggingface.co/datasets)

### Community & Learning

*   **Hugging Face Course:** A free, comprehensive course to get started with 🤗 Transformers.
    *   [https://huggingface.co/course](https://huggingface.co/course)
*   **Hugging Face Blog:** Stay updated with the latest news, tutorials, and research from the Hugging Face team and community.
    *   [https://huggingface.co/blog](https://huggingface.co/blog)
*   **Hugging Face Forum:** Ask questions, share knowledge, and connect with other users and developers.
    *   [https://discuss.huggingface.co/](https://discuss.huggingface.co/)

### Exploring the Hub

Don't forget to explore the [Hugging Face Hub](https://huggingface.co/models) itself! You can filter models by task, language, and other criteria to find exactly what you need. Each model page provides an interactive inference widget, code snippets, and details about the model's architecture and training.